# Flood Depth / Extent Difference Maps (scenario vs baseline)

Builds three results maps from a pair of max-flood-depth GeoTIFFs (one being evaluated,
one treated as the baseline/reference it's compared against). **Pick one baseline and any number
of scenarios to evaluate against it** (`SIM_PAIRS`) -- everything below runs once per
pair, in one pass, so you don't have to re-run the (slow) raster processing by hand for each
scenario separately.

1. **Flood depth difference map** -- continuous, `compare - baseline`. Negative (green) means
   the compared scenario made it shallower there -- good. Positive (red) means it made it
   deeper there -- bad. Marked with two dots: an olive dot at the single cell where depth was
   reduced the most (the biggest benefit anywhere), and a small orange dot at the deepest cell
   among newly-flooded ones (the worst new impact).
2. **Flood extent difference map** -- categorical: wet in both (dark blue), avoided flooding
   thanks to the compared scenario (grey), new flooding caused by it (fuchsia).
3. **Combined map** -- grey/red plain colours for the two extent-change categories
   (avoided / new), with the "wet in both" class shaded on a continuous green-to-red
   depth-difference gradient instead of a flat colour.
4. **Binned depth-difference map** -- the signed change classified into depth-change classes
   (unchanged from before).

**Also computed, per pair**:
- **Table A** -- total flooded area (depth > `WET_THRESHOLD_M`): before (baseline), after
  (compare), the absolute difference, and the percentage difference.
- **Table B** -- the SAME comparison, but split into depth-magnitude bins (0.1-0.2 m, 0.2-0.5 m,
  0.5-1 m, 1-2 m, 2-5 m, > 5 m): how much area falls in each bin under the baseline, under the
  compared scenario, and the difference/percentage between them. Computed in the SAME
  block-by-block pass that writes the three GeoTIFFs, so it costs no extra raster read.

Every table, print statement, map title, legend and output filename is driven by the baseline +
compare tags automatically -- nothing needs manual renaming per pair, and every plot renders in
Times New Roman with every font size a named, adjustable constant.

Three things this deliberately avoids:

- **QGIS's Raster Calculator NoData-masking bug** -- any formula involving a NoData
  pixel returns NoData regardless of IF/AND logic. Here, NoData is converted to an
  explicit depth of 0.0 in Python before any arithmetic happens -- dry means zero, not
  "unknown" -- so subtraction and comparisons behave exactly as expected.
- **Loading the whole domain into memory at once** -- your merged/downscaled depth
  rasters cover the full domain at fine resolution (tens of thousands of pixels per
  side), so a single full-resolution array can be several GB. Every full-resolution
  step below processes the raster **block by block** and writes each block straight to
  the output GeoTIFF, so peak memory stays small and constant no matter how big the
  domain is. The only full-domain arrays ever held in memory are small, *decimated*
  (downsampled) previews used purely for picking a colour scale and showing you a
  quick-look plot.
- **Upside-down previews on south-up rasters** -- some rasters store
  rows south-first (a positive Y pixel size in the transform) rather than the usual
  north-first convention. The preview plots detect this from the transform and flip
  the display orientation automatically, so the quick-look PNGs match the real map
  regardless of which convention the source file uses. (This never affected the actual
  output GeoTIFFs -- GIS software always reads the transform directly -- only the
  matplotlib previews.)

**Input:** one **maximum flood depth raster** (GeoTIFF, metres) per scenario -- the usual "max
depth" output of any 2D flood model.

**To reuse for another scenario set**: this notebook is built around a `SIM_MAPS` library of
every simulation you have (or will have) available, keyed by short tags of your choosing -- the
example uses `BL` (baseline) and `S1`, `S2`, `S3` (scenarios). Add each scenario's raster path to
`SIM_MAPS` as it becomes available, then list the `(baseline, scenario)` pairs you want compared
in this run in `SIM_PAIRS`. Several events? Give each its own tags (e.g. `BL_T100`, `S1_T100`)
and pair each scenario with the baseline of the same event.


In [ ]:
import csv
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import rasterio
from rasterio.warp import Resampling
from rasterio.vrt import WarpedVRT


## Config

- `SIM_MAPS`: the full library of simulation rasters available for comparison. Add a scenario
  here (as a `"KEY": r"path"` entry) as soon as its raster exists -- entries left as `""` are
  scenarios that haven't been run yet. The keys are free-form tags; the example uses `BL` for the
  baseline and `S1`/`S2`/`S3` for the scenarios.
- `SIM_PAIRS`: every `(baseline, scenario)` pair to evaluate in this run -- the baseline is the
  "without" / reference case. Each pair gets its own three maps, Table A, Table B, binned map
  and CSVs, all tagged so they never overwrite each other. Change this list to compare a
  different set; every print statement, map title, legend, stats table, and output filename is
  driven by it automatically.
- `WET_THRESHOLD_M`: minimum depth (m) for a cell to count as "flooded" -- 0.10 m by default
  (a 5 cm cutoff tends to be noisy).
- `DEPTH_MAG_BIN_EDGES` / `DEPTH_MAG_BIN_LABELS`: the depth-magnitude classes for Table B (how
  much area falls in each bin, baseline vs compare) -- 0.1-0.2 m, 0.2-0.5 m, 0.5-1 m, 1-2 m,
  2-5 m, and an open-ended `> 5 m` class.
- `DEPTH_DIFF_VMAX_M`: the +/- range of the depth-difference colour scale (`compare -
  baseline`). Leave as `None` to auto-pick it from a cheap downsampled preview (98th
  percentile of |difference|), or set a fixed number (e.g. `1.0`) to keep the same
  scale across every scenario pair you compare, which makes figures easier to compare side
  by side.
- `PREVIEW_DOWNSAMPLE`: how much to shrink the raster for the quick-look preview plots
  and the auto vmax estimate. 25 means the preview is 1/25th resolution in each
  dimension (1/625th the pixel count) -- plenty for picking a colour scale and eyeballing
  the result; the actual GeoTIFFs written to disk are always full resolution.
- `OUTPUT_DIR`: where the output GeoTIFFs + PNG previews + summary/table CSVs get written.
- `DIFF_CMAP`: the diverging colour map used for the depth-difference gradient (Map 1,
  and the "wet in both" cells of Map 3). A custom green-white-red map: green on the
  negative (good/depth-reduced) side, red on the positive (bad/depth-increased) side.
- `MAP2_COLORS` / `MAP3_PLAIN_COLORS`: the fixed, non-gradient colours for each map's
  categorical classes (`0`=dry in both, `1`=wet in both, `2`=avoided thanks to the compared
  scenario, `3`=new flooding caused by it). Map 3 has no entry for `1` because that class is
  drawn as a gradient there instead of a flat colour.
- `DOT_MAX_AVOIDED_COLOR` / `DOT_MAX_NEW_DEPTH_COLOR` (+ their `_SIZE`s): the two marker dots on
  the depth-difference map -- max depth avoided (olive, larger) and max new depth (orange,
  smaller) -- and the fonts block below it (Times New Roman, every size a named constant).


In [ ]:
# Library of every simulation raster you have (or will have) available. Add new scenarios
# here as their runs finish -- leave the path as "" for a scenario that doesn't exist yet.
# Keys are short tags of your choosing; they appear in every title, table and file name.
SIM_MAPS = {
    #"BL": r"path/to/flood_maps/BL_max_depth.tif",    # baseline
    #"S1": r"path/to/flood_maps/S1_max_depth.tif",    # scenario 1
}

# Every (baseline, compare) pair to process in this run. Each pair gets its own three maps,
# binned map, Table A, Table B and CSVs -- all pairs run in one go, no re-editing between them.
# Both tags of a pair must be keys in SIM_MAPS above with a real (non-empty) path. Several
# events? Pair each scenario with its own event's baseline, e.g. ("BL_T100", "S1_T100").
SIM_PAIRS = [
    ("BL", "S1"),
]


def _require_map(tag):
    path = SIM_MAPS.get(tag, "")
    if not path:
        raise ValueError(f"SIM_MAPS['{tag}'] is empty -- fill in that scenario's raster path "
                         f"before using it.")
    return path


for _b, _c in SIM_PAIRS:
    _require_map(_b)
    _require_map(_c)

WET_THRESHOLD_M = 0.10

# Depth-MAGNITUDE bins for Table B (how much area falls in each depth class, baseline vs
# compare) -- NOT the signed change classes used by the binned depth-difference map further
# down, which is a separate, already-existing feature left as-is. Each bin is [lower, upper);
# the last is open-ended.
DEPTH_MAG_BIN_EDGES  = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, np.inf]
DEPTH_MAG_BIN_LABELS = ["0.1 - 0.2", "0.2 - 0.5", "0.5 - 1", "1 - 2", "2 - 5", "> 5"]

# ---------------------------------------------------------------------------
# FIXED COLOUR SCALE -- this is what makes every run comparable.
#
# This single number sets the limits of the green-white-red depth-difference
# ramp on: the Map 1 preview, the Map 3 preview + colourbar, AND the full
# resolution combined_diff GeoTIFF. Set it once, never change it, and every
# scenario pair you produce can be laid side by side.
#
# If this is None the notebook picks the 98th percentile of whatever pair is
# loaded, so each run gets a different ramp and the previews are NOT
# comparable.
DEPTH_DIFF_VMAX_M = 1.0        # metres. Same for all scenario pairs.

STRICT_FIXED_SCALE = True      # True  = refuse to run if the scale is not pinned
                               # False = allow auto-scaling, with a loud warning
PREVIEW_DOWNSAMPLE = 25
DIFF_CMAP = mcolors.LinearSegmentedColormap.from_list(
    "GreenWhiteRed", ["#1a9641", "#ffffff", "#d7191c"]
)  # green = negative (compare - baseline < 0, shallower, good)
   # red   = positive (compare - baseline > 0, deeper, bad)

OUTPUT_DIR = Path(r"path/to/output_dir")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Map 2 -- flood extent difference (every class is a flat colour)
MAP2_COLORS = {
    0: (1.00, 1.00, 1.00, 0.00),  # dry both -- transparent
    1: (0.05, 0.10, 0.45, 0.85),  # wet both -- dark blue
    2: (0.55, 0.55, 0.55, 0.80),  # avoided (wet only in baseline) -- grey
    3: (1.00, 0.00, 1.00, 0.90),  # new (wet only in compare) -- fuchsia
}

# Map 3 -- combined map. Only the plain (non-gradient) classes go here;
# class 1 ("wet in both") is drawn with the DIFF_CMAP gradient instead.
MAP3_PLAIN_COLORS = {
    0: (1.00, 1.00, 1.00, 0.00),  # dry both -- transparent
    2: (0.55, 0.55, 0.55, 0.85),  # avoided (wet only in baseline) -- grey, plain
    3: (0.85, 0.05, 0.05, 0.90),  # new (wet only in compare) -- red, plain
}

# ---------------- MARKER DOTS on the depth-difference map ----------------
DOT_MAX_AVOIDED_COLOR   = "olive"   # single cell with the biggest depth REDUCTION anywhere
DOT_MAX_AVOIDED_SIZE    = 11        # marker size (points) -- deliberately the bigger/more visible dot
DOT_MAX_NEW_DEPTH_COLOR = "orange"  # deepest cell among NEWLY flooded cells only
DOT_MAX_NEW_DEPTH_SIZE  = 5         # marker size (points) -- deliberately small, per spec

# ================= FONTS -- Times New Roman, EVERY size adjustable =================
# Exactly which text each knob drives. Every piece of text in every figure this notebook draws
# is covered by one of these -- nothing is hardcoded any more.
#
#   FS_TITLE         the title line on top of ALL FOUR maps (depth, extent, combined, binned)
#   FS_AXIS_LABEL    the colourbar labels on the depth map and the combined map
#   FS_TICKS         the number labels along those colourbars
#   FS_LEGEND        the legend ENTRY text: extent classes, combined-map classes, the depth map's
#                    two marker dots, and the binned map's class list
#   FS_LEGEND_TITLE  the legend's heading word, where a legend has one ("Depth change")
#   FS_STATS_TABLE   the little stats box in the bottom-left corner of the depth/extent/combined maps
#
# AFTER EDITING: re-run THIS cell, then re-run the map cells. Editing a number here and re-running
# only the map cell picks the change up fine, but editing it and re-running nothing does not --
# that is the usual reason a size "doesn't change". The printout at the bottom of this cell
# echoes the values actually in force, so you can check at a glance.
FONT_FAMILY   = "Times New Roman"   # falls back to a serif face if not installed
FS_TITLE        = 20   # #FONTSIZE
FS_AXIS_LABEL   = 14   # #FONTSIZE
FS_TICKS        = 12   # #FONTSIZE
FS_LEGEND       = 18   # #FONTSIZE
FS_LEGEND_TITLE = 16   # #FONTSIZE
FS_STATS_TABLE  = 15   # #FONTSIZE

MAP_FIGSIZE = (9, 8)   # #LAYOUT -- the size (inches) of each map figure. If the text feels small
                       # NEXT TO THE MAP, this is the knob that fixes it, not the font sizes:
                       # raising a font size makes the text bigger in points, but the map then has
                       # to share the same canvas with it. Raise both together (e.g. (13, 11) with
                       # the bigger fonts) to keep the map large AND the labels readable.
MAP_LEGEND_RIGHT = 0.74  # #LAYOUT -- for the extent map, the share of the figure width the MAP
                         # keeps; the rest is reserved for the legend on its right. Lower it if a
                         # long legend crowds the map.

_avail = {f.name for f in mpl.font_manager.fontManager.ttflist}
if FONT_FAMILY in _avail:
    mpl.rcParams["font.family"] = FONT_FAMILY
else:
    _alt = next((a for a in ("Liberation Serif", "DejaVu Serif", "Nimbus Roman")
                if a in _avail), "serif")
    mpl.rcParams["font.family"] = _alt
    print(f"!! '{FONT_FAMILY}' not installed -- using '{_alt}'")
mpl.rcParams["mathtext.fontset"] = "dejavuserif"
# Belt and braces: these make the rcParams defaults follow the same knobs, so any text element
# that ever gets added without an explicit fontsize= still scales with them instead of silently
# staying at matplotlib's 10pt default.
mpl.rcParams.update({
    "font.size":         FS_STATS_TABLE,
    "axes.titlesize":    FS_TITLE,
    "axes.labelsize":    FS_AXIS_LABEL,
    "xtick.labelsize":   FS_TICKS,
    "ytick.labelsize":   FS_TICKS,
    "legend.fontsize":   FS_LEGEND,
    "legend.title_fontsize": FS_LEGEND_TITLE,
    "figure.titlesize":  FS_TITLE,
})

print(f"{len(SIM_PAIRS)} pair(s) to process this run:")
for _b, _c in SIM_PAIRS:
    print(f"  {_b} vs {_c}")
print(f"\nFont sizes in force -- title {FS_TITLE}, colourbar label {FS_AXIS_LABEL}, "
      f"colourbar ticks {FS_TICKS}, legend {FS_LEGEND}, legend title {FS_LEGEND_TITLE}, "
      f"stats box {FS_STATS_TABLE}   (family: {mpl.rcParams['font.family'][0]})")


## Step 0 -- Grid audit + colour-scale calibration (run once)

Two things have to be identical across scenario pairs for the previews to be
comparable:

1. **The grid.** If two rasters have different bounds or sizes, their preview
   images cover different ground, so the pictures are not comparable even if the
   colours are.
2. **The colour scale.** `DEPTH_DIFF_VMAX_M` must be a fixed number, not `None`.

This cell checks (1) and gives you a defensible number for (2). Run it once,
paste the suggested value into the config cell, and never touch it again.


In [ ]:
# ============================================================
# Step 0 -- run ONCE, then paste the suggested DEPTH_DIFF_VMAX_M into the config
# ============================================================

# The scenario pairs you actually intend to present. The colour scale must cover
# all of them, so list every pair that will appear in a report.
CALIBRATION_PAIRS = [
    ("BL", "S1"),
    ("BL", "S2"),
    ("BL", "S3"),
]

CALIB_DOWNSAMPLE = 50     # coarser than the preview -- this is only for percentiles
CALIB_PCT = 98.0          # percentile of |difference| the scale should reach

# ---- part 1: grid audit -------------------------------------------------
print("GRID AUDIT")
print("-" * 78)
grids = {}
for key, path in SIM_MAPS.items():
    if not path:
        continue
    try:
        with rasterio.open(path) as s:
            sig = (s.height, s.width, tuple(round(v, 6) for v in s.transform[:6]), str(s.crs))
            grids.setdefault(sig, []).append(key)
    except Exception as e:
        print(f"  {key:<18} COULD NOT OPEN: {e}")

if len(grids) == 1:
    sig = next(iter(grids))
    print(f"  All {sum(len(v) for v in grids.values())} maps share one grid: "
          f"{sig[0]} x {sig[1]}, {sig[3]}")
    print("  -> previews will cover identical ground. Good.")
else:
    print(f"  !! {len(grids)} DIFFERENT GRIDS FOUND. Previews will not be comparable:")
    for sig, keys in grids.items():
        print(f"     {sig[0]:>6} x {sig[1]:<6} {sig[3]:<12} : {', '.join(keys)}")
    print("  -> the notebook reprojects the baseline onto the compare grid, so the")
    print("     numbers stay correct, but the preview EXTENT changes between pairs.")
    print("     Regenerate the odd ones out on the common grid before making figures.")

# ---- part 2: colour-scale calibration -----------------------------------
print()
print("COLOUR SCALE CALIBRATION")
print("-" * 78)
print(f"  {'pair':<28}{'p' + str(CALIB_PCT):>9}{'p99.8':>9}{'max':>9}")
needed = []
for base_key, cmp_key in CALIBRATION_PAIRS:
    bp, cp = SIM_MAPS.get(base_key, ""), SIM_MAPS.get(cmp_key, "")
    if not bp or not cp:
        print(f"  {base_key + ' vs ' + cmp_key:<28}  (skipped -- path not set)")
        continue
    try:
        with rasterio.open(cp) as s:
            h = max(1, s.height // CALIB_DOWNSAMPLE)
            w = max(1, s.width // CALIB_DOWNSAMPLE)
            a = s.read(1, out_shape=(h, w), resampling=Resampling.average,
                      masked=True).filled(0.0)
        with rasterio.open(bp) as s:
            b = s.read(1, out_shape=(h, w), resampling=Resampling.average,
                      masked=True).filled(0.0)
    except Exception as e:
        print(f"  {base_key + ' vs ' + cmp_key:<28}  ERROR: {e}")
        continue
    a = np.where(a < 0, 0.0, a).astype("float32")
    b = np.where(b < 0, 0.0, b).astype("float32")
    valid = (a > WET_THRESHOLD_M) | (b > WET_THRESHOLD_M)
    d = np.abs(np.where(valid, a - b, np.nan))
    if not np.isfinite(d).any():
        print(f"  {base_key + ' vs ' + cmp_key:<28}  (no overlapping wet area)")
        continue
    p = float(np.nanpercentile(d, CALIB_PCT))
    p998 = float(np.nanpercentile(d, 99.8))
    mx = float(np.nanmax(d))
    needed.append(p)
    print(f"  {base_key + ' vs ' + cmp_key:<28}{p:>9.2f}{p998:>9.2f}{mx:>9.2f}")

if needed:
    raw = max(needed)
    nice = next((v for v in [0.25, 0.5, 0.75, 1.0, 1.5, 2.0, 3.0, 5.0] if v >= raw), 5.0)
    print()
    print(f"  Largest p{CALIB_PCT} across all pairs: {raw:.2f} m")
    print(f"  ==> set  DEPTH_DIFF_VMAX_M = {nice}  in the config cell, and leave it there.")
    print()
    print("  Rationale for your methods: the scale is fixed across all scenario")
    print(f"  comparisons and covers the {CALIB_PCT}th percentile of the largest depth")
    print("  change observed in any pair, so the colours mean the same thing on")
    print("  every figure.")


## Steps 1-4 -- the per-pair pipeline

Everything that used to be four separate cells (open + align, downsampled preview + colour
scale, the full-resolution block pass, and the statistics) is now one function,
`process_scenario_pair(baseline_tag, compare_tag)`, so it can run once per pair in
`SIM_PAIRS` without copy-pasting cells or re-editing config between runs.

**Step 1 -- open + align.** If the two rasters already share the exact same grid (same domain,
same quadtree refinement -- typical when the compared scenario only changes land use/roughness,
not the grid), the baseline raster is read directly. If they don't line up, it's wrapped in a
`WarpedVRT` -- a *virtual*, lazily-reprojected view aligned to the compare-scenario grid, so
nothing gets resampled into memory up front; blocks only get reprojected on demand as the main
loop reads them. This is also where the south-up check happens (`plot_origin`, from the sign of
the transform's Y pixel size), so every preview later renders right-side-up regardless of which
convention the source file uses.

**Step 2 -- cheap downsampled preview.** Reads both rasters at `1/PREVIEW_DOWNSAMPLE` resolution
directly from disk (a decimated read -- GDAL only reads roughly one out of every
`PREVIEW_DOWNSAMPLE` pixels, it doesn't read everything and then shrink it), used for the auto
vmax and the quick-look plots; the full-resolution GeoTIFFs are written separately in Step 3.

**Step 3 -- the only full-resolution pass**, never holding more than one block's worth of pixels
in memory. For each block it writes the depth difference, the extent category, and the combined
RGBA colour straight to their output files, AND accumulates every running statistic needed
afterwards in the same pass: wet-cell counts, the max/min single-cell difference (with location),
the max depth among newly-flooded cells (with location), the sum/count for average depth
reduction, and -- new -- how many cells fall in each `DEPTH_MAG_BIN_EDGES` class for the baseline
and for the compared scenario (Table B). No second full-resolution pass is needed for anything.

**Step 4 -- statistics and both tables.**
- *Area avoided* / *area newly flooded*: the extent-change footprints (Map 2's stats).
- *Average / max depth avoided*, *max new depth*: the depth-change stats (Map 1's stats) -- see
  the docstring on `process_scenario_pair` for exactly how each is defined (unchanged from
  before). The depth map's on-map table no longer prints coordinates for these two points --
  they're marked directly on the map instead (the olive and orange dots).
- **Table A**: total flooded area, before (baseline) vs after (compare), the absolute difference,
  and the percentage difference.
- **Table B**: the same before/after/diff/%-diff comparison, split into the `DEPTH_MAG_BIN_EDGES`
  depth classes.

The combined map's on-map table keeps only the two headline numbers (area avoided, average depth
avoided) -- area newly flooded, max new depth and max depth avoided are dropped there (they're
still fully reported on Maps 1 and 2, and in the CSVs).


In [ ]:
def read_block_zero_filled(src, window):
    """Read one window and treat NoData / negative sentinels as 0 m depth (dry)."""
    arr = src.read(1, window=window, masked=True).filled(0.0).astype("float32")
    return np.where(arr < 0, 0.0, arr)


def _depth_bin_index(depth_arr):
    """Depth array -> bin index 0..len(DEPTH_MAG_BIN_LABELS)-1, or -1 for dry (< first edge)."""
    idx = np.full(depth_arr.shape, -1, dtype="int8")
    lo = DEPTH_MAG_BIN_EDGES[0]
    for k, hi in enumerate(DEPTH_MAG_BIN_EDGES[1:]):
        sel = (depth_arr >= lo) & (depth_arr < hi) if np.isfinite(hi) else (depth_arr >= lo)
        idx[sel] = k
        lo = hi
    return idx


def add_stats_table(ax, text, fontsize=None):
    """Draw a stats-table text block in the corner of a map."""
    ax.text(
        0.02, 0.02, text, transform=ax.transAxes,
        fontsize=FS_STATS_TABLE if fontsize is None else fontsize, va="bottom", ha="left",
        color="black",
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white", edgecolor="black", alpha=0.85),
    )


def process_scenario_pair(baseline_tag, compare_tag):
    """Runs the full depth/extent comparison pipeline for one baseline-vs-compare pair (see the
    markdown above for what each step does). Returns (table_a_row, table_b_df) so a driver cell
    can combine results across every scenario processed in one run.

    Depth-stat definitions (unchanged from the original single-pair version):
      average depth avoided : mean depth reduction, per cell, over cells flooded in BOTH
                               scenarios where the compared scenario reduced the depth there
                               (excludes fully-avoided cells -- a different kind of benefit,
                               already captured by the extent stats).
      max depth avoided     : the single largest depth REDUCTION anywhere in the domain, whether
                               that cell ended up merely shallower (still wet in both) or fully
                               dry (avoided) -- the biggest single benefit. Marked with the olive
                               dot on Map 1.
      max new depth         : the deepest a cell gets among cells completely dry in the baseline
                               that only became wet because of the compared scenario -- a
                               genuinely different population from "max depth avoided" (it's the
                               compared scenario's own depth value at a newly-wet cell, not a
                               difference against a previously-wet baseline). Marked with the
                               small orange dot on Map 1.
    """
    sc_tif = _require_map(compare_tag)
    bl_tif = _require_map(baseline_tag)
    scenario_tag = f"{baseline_tag}_vs_{compare_tag}"
    diff_out = OUTPUT_DIR / f"depth_diff_{scenario_tag}.tif"
    class_out = OUTPUT_DIR / f"extent_diff_{scenario_tag}.tif"
    combined_out = OUTPUT_DIR / f"combined_diff_{scenario_tag}.tif"

    print(f"\n{'=' * 78}\nComparing '{compare_tag}' against baseline '{baseline_tag}'  (tag: {scenario_tag})\n{'=' * 78}")
    print(f"  {baseline_tag}: {bl_tif}")
    print(f"  {compare_tag}: {sc_tif}")

    # ---- Step 1: open + align ----
    sc_src = rasterio.open(sc_tif)
    bl_src_raw = rasterio.open(bl_tif)
    same_grid = (sc_src.transform == bl_src_raw.transform and sc_src.crs == bl_src_raw.crs
                and sc_src.width == bl_src_raw.width and sc_src.height == bl_src_raw.height)
    if same_grid:
        print(f"Grids match exactly -- reading {baseline_tag} raster directly, no reprojection needed.")
        bl_src = bl_src_raw
    else:
        print(f"Grids differ -- wrapping {baseline_tag} raster in a WarpedVRT aligned to the {compare_tag} grid.")
        bl_src = WarpedVRT(bl_src_raw, crs=sc_src.crs, transform=sc_src.transform,
                              width=sc_src.width, height=sc_src.height, resampling=Resampling.bilinear)

    transform, crs = sc_src.transform, sc_src.crs
    width, height = sc_src.width, sc_src.height
    print(f"Grid: {width} x {height} ({width * height:,} pixels), resolution {transform.a:.2f} m")
    plot_origin = "upper" if transform.e < 0 else "lower"
    print(f"Transform Y pixel size: {transform.e:.4f} -> using imshow origin='{plot_origin}'")

    try:
        # ---- Step 2: cheap downsampled preview + colour scale ----
        small_h = max(1, height // PREVIEW_DOWNSAMPLE)
        small_w = max(1, width // PREVIEW_DOWNSAMPLE)
        sc_small = sc_src.read(1, out_shape=(small_h, small_w), resampling=Resampling.average, masked=True).filled(0.0)
        bl_small = bl_src.read(1, out_shape=(small_h, small_w), resampling=Resampling.average, masked=True).filled(0.0)
        sc_small = np.where(sc_small < 0, 0.0, sc_small).astype("float32")
        bl_small = np.where(bl_small < 0, 0.0, bl_small).astype("float32")

        print(f"Preview grid: {small_w} x {small_h} ({small_w * small_h:,} pixels)")
        print(f"{compare_tag} depth range (preview):    {sc_small.min():.3f} - {sc_small.max():.3f} m")
        print(f"{baseline_tag} depth range (preview): {bl_small.min():.3f} - {bl_small.max():.3f} m")

        wet_sc_small = sc_small > WET_THRESHOLD_M
        wet_bl_small = bl_small > WET_THRESHOLD_M
        either_wet_small = wet_sc_small | wet_bl_small
        diff_small = sc_small - bl_small
        diff_small_masked = np.where(either_wet_small, diff_small, np.nan)

        _absdiff = np.abs(diff_small_masked)
        _n_valid = int(np.isfinite(_absdiff).sum())
        _p98 = float(np.nanpercentile(_absdiff, 98)) if _n_valid else 0.0
        if DEPTH_DIFF_VMAX_M is None:
            if STRICT_FIXED_SCALE:
                raise ValueError(
                    f"DEPTH_DIFF_VMAX_M is None -- this pair ({scenario_tag}) would auto-scale to "
                    f"+/-{_p98:.2f} m, making it not comparable with other pairs. Set a fixed "
                    f"number in the config cell (run Step 0 to choose one) or set "
                    f"STRICT_FIXED_SCALE=False if you really want a throwaway auto-scaled look.")
            vmax = max(_p98, 0.05)
            print("!! WARNING: auto-scaled colour ramp. Not comparable with any other pair.")
        else:
            vmax = float(DEPTH_DIFF_VMAX_M)
        print(f"Depth-difference colour scale: +/- {vmax:.2f} m"
              f"{'  [FIXED]' if DEPTH_DIFF_VMAX_M is not None else '  [AUTO -- not comparable]'}")
        cmap = DIFF_CMAP
        norm = mcolors.TwoSlopeNorm(vmin=-vmax, vcenter=0, vmax=vmax)

        # ---- Step 3: full-resolution block loop -- writes 3 GeoTIFFs AND bins depth magnitudes ----
        diff_profile = dict(driver="GTiff", height=height, width=width, count=1, dtype="float32",
                            crs=crs, transform=transform, nodata=np.nan, compress="deflate",
                            tiled=True, blockxsize=256, blockysize=256, BIGTIFF="YES")
        class_profile = dict(driver="GTiff", height=height, width=width, count=1, dtype="uint8",
                             crs=crs, transform=transform, nodata=0, compress="deflate",
                             tiled=True, blockxsize=256, blockysize=256, BIGTIFF="YES")
        rgba_profile = dict(driver="GTiff", height=height, width=width, count=4, dtype="uint8",
                            crs=crs, transform=transform, compress="deflate", photometric="RGB",
                            alpha="yes", tiled=True, blockxsize=256, blockysize=256, BIGTIFF="YES")

        wet_sc_count = wet_bl_count = wet_both_count = avoided_count = new_count = 0
        n_blocks = 0
        sum_depth_reduced = 0.0
        count_depth_reduced = 0
        global_max_diff, global_max_rc = -np.inf, None
        global_min_diff, global_min_rc = np.inf, None
        global_new_max_depth, global_new_max_rc = -np.inf, None

        n_bins = len(DEPTH_MAG_BIN_LABELS)
        baseline_bin_counts = np.zeros(n_bins, dtype="int64")
        compare_bin_counts = np.zeros(n_bins, dtype="int64")

        with rasterio.open(diff_out, "w", **diff_profile) as diff_dst, \
             rasterio.open(class_out, "w", **class_profile) as class_dst, \
             rasterio.open(combined_out, "w", **rgba_profile) as rgba_dst:

            for _, window in sc_src.block_windows(1):
                sc_b = read_block_zero_filled(sc_src, window)
                bl_b = read_block_zero_filled(bl_src, window)

                wet_sc_b = sc_b > WET_THRESHOLD_M
                wet_bl_b = bl_b > WET_THRESHOLD_M
                either_wet_b = wet_sc_b | wet_bl_b
                wet_both_b = wet_sc_b & wet_bl_b
                avoided_b = wet_bl_b & ~wet_sc_b   # wet in baseline, dry in compare -- avoided
                new_b = wet_sc_b & ~wet_bl_b       # dry in baseline, wet in compare -- new flooding

                # --- Map 1: depth difference (compare - baseline) ---
                diff_b = sc_b - bl_b
                diff_masked_b = np.where(either_wet_b, diff_b, np.nan).astype("float32")
                diff_dst.write(diff_masked_b, 1, window=window)

                # --- Map 2: extent category ---
                class_b = np.zeros(sc_b.shape, dtype="uint8")
                class_b[wet_both_b] = 1
                class_b[avoided_b] = 2
                class_b[new_b] = 3
                class_dst.write(class_b, 1, window=window)

                # --- Map 3: combined RGBA -- gradient for "wet both", plain colours elsewhere ---
                rgba_b = cmap(norm(diff_b)).astype("float32")
                rgba_b[..., 3] = 0.85
                rgba_b[class_b == 0] = MAP3_PLAIN_COLORS[0]
                rgba_b[class_b == 2] = MAP3_PLAIN_COLORS[2]
                rgba_b[class_b == 3] = MAP3_PLAIN_COLORS[3]
                rgba_uint8_b = np.clip(rgba_b * 255, 0, 255).astype("uint8")
                for b in range(4):
                    rgba_dst.write(rgba_uint8_b[:, :, b], b + 1, window=window)

                # --- running stats ---
                wet_sc_count += int(wet_sc_b.sum())
                wet_bl_count += int(wet_bl_b.sum())
                wet_both_count += int(wet_both_b.sum())
                avoided_count += int(avoided_b.sum())
                new_count += int(new_b.sum())
                n_blocks += 1

                reduced_b = wet_both_b & (diff_b < 0)
                if reduced_b.any():
                    sum_depth_reduced += float(np.abs(diff_b[reduced_b]).sum())
                    count_depth_reduced += int(reduced_b.sum())

                if either_wet_b.any():
                    stat_field = np.where(either_wet_b, diff_b, np.nan)
                    local_max_idx = np.nanargmax(stat_field)
                    local_max_val = float(stat_field.flat[local_max_idx])
                    if local_max_val > global_max_diff:
                        r, c = np.unravel_index(local_max_idx, diff_b.shape)
                        global_max_diff = local_max_val
                        global_max_rc = (window.row_off + r, window.col_off + c)

                    local_min_idx = np.nanargmin(stat_field)
                    local_min_val = float(stat_field.flat[local_min_idx])
                    if local_min_val < global_min_diff:
                        r, c = np.unravel_index(local_min_idx, diff_b.shape)
                        global_min_diff = local_min_val
                        global_min_rc = (window.row_off + r, window.col_off + c)

                if new_b.any():
                    new_depth_field = np.where(new_b, sc_b, -np.inf)
                    local_new_max_idx = np.argmax(new_depth_field)
                    local_new_max_val = float(new_depth_field.flat[local_new_max_idx])
                    if local_new_max_val > global_new_max_depth:
                        r, c = np.unravel_index(local_new_max_idx, sc_b.shape)
                        global_new_max_depth = local_new_max_val
                        global_new_max_rc = (window.row_off + r, window.col_off + c)

                # depth-magnitude bin accumulation (Table B) -- reuses these SAME blocks, no extra I/O
                b_idx = _depth_bin_index(bl_b)
                c_idx = _depth_bin_index(sc_b)
                for k in range(n_bins):
                    baseline_bin_counts[k] += int((b_idx == k).sum())
                    compare_bin_counts[k] += int((c_idx == k).sum())
    finally:
        sc_src.close()
        bl_src_raw.close()
        if not same_grid:
            bl_src.close()

    cell_area_m2 = abs(transform.a * transform.e)
    print(f"Processed {n_blocks:,} blocks.")
    print(f"Wet cells, {compare_tag}: {wet_sc_count:,}  ({wet_sc_count * cell_area_m2 / 1e6:.3f} km2)")
    print(f"Wet cells, {baseline_tag}: {wet_bl_count:,}  ({wet_bl_count * cell_area_m2 / 1e6:.3f} km2)")
    print(f"Wet in both: {wet_both_count:,}")
    print(f"New dry cells in {compare_tag}: {avoided_count:,}  ({avoided_count * cell_area_m2 / 1e6:.3f} km2)")
    print(f"New wet cells in {compare_tag}: {new_count:,}  ({new_count * cell_area_m2 / 1e6:.3f} km2)")
    print(f"Saved: {diff_out}\nSaved: {class_out}\nSaved: {combined_out}")

    # ---- Step 4: statistics ----
    avg_depth_reduced = sum_depth_reduced / count_depth_reduced if count_depth_reduced else float("nan")
    area_avoided_km2 = avoided_count * cell_area_m2 / 1e6
    area_new_km2 = new_count * cell_area_m2 / 1e6
    pct_avoided_of_baseline = 100 * avoided_count / wet_bl_count if wet_bl_count else float("nan")

    def rowcol_to_xy(rc):
        if rc is None:
            return None
        r, c = rc
        return rasterio.transform.xy(transform, r, c)

    max_xy = rowcol_to_xy(global_max_rc)
    min_xy = rowcol_to_xy(global_min_rc)          # "max depth avoided" point -- olive dot
    new_max_xy = rowcol_to_xy(global_new_max_rc)   # "max new depth" point -- orange dot
    max_depth_avoided_m = abs(global_min_diff)
    max_new_depth_m = global_new_max_depth if global_new_max_rc is not None else float("nan")

    print("\n[Extent]")
    print(f"Area avoided ({compare_tag} dry, {baseline_tag} wet): {area_avoided_km2:.4f} km2"
          f"  =  {pct_avoided_of_baseline:.1f}% of {baseline_tag}'s wet area")
    print(f"Area newly flooded ({compare_tag} wet, {baseline_tag} dry): {area_new_km2:.4f} km2")
    print("\n[Depth]")
    print(f"Average depth avoided (wet-in-both cells, {compare_tag} reduced): {avg_depth_reduced:.3f} m  (n={count_depth_reduced:,} cells)")
    print(f"Max depth avoided (single best cell, anywhere): {max_depth_avoided_m:.3f} m   x/y ({crs}): {min_xy}")
    print(f"Max new depth (single deepest newly-flooded cell): {max_new_depth_m:.3f} m   x/y ({crs}): {new_max_xy}")
    print(f"(For reference -- single worst cell overall, {compare_tag} increased depth most: "
          f"{global_max_diff:+.3f} m at x/y {max_xy})")

    # STATS_TEXT_DEPTH -- no coordinates here any more; the two points are marked on the map
    # itself instead (the olive + orange dots), so the on-map table only needs the values.
    STATS_TEXT_DEPTH = (
        f"Avg depth avoided (wet-in-both): {avg_depth_reduced:.3f} m\n"
        f"Max depth avoided: {max_depth_avoided_m:.3f} m\n"
        f"Max new depth: {max_new_depth_m:.3f} m"
    )
    STATS_TEXT_EXTENT = (
        f"Area avoided: {area_avoided_km2:.3f} km2 ({pct_avoided_of_baseline:.1f}% of {baseline_tag} wet area)\n"
        f"Area newly flooded: {area_new_km2:.3f} km2"
    )
    # Combined map's table keeps only these two headline numbers -- area newly flooded, max new
    # depth and max depth avoided are dropped here (still fully reported on Maps 1 and 2, and CSVs).
    STATS_TEXT_COMBINED = (
        f"Area avoided: {area_avoided_km2:.3f} km2 ({pct_avoided_of_baseline:.1f}% of {baseline_tag} wet area)\n"
        f"Avg depth avoided (wet-in-both): {avg_depth_reduced:.3f} m"
    )

    # ---- Table A: total flooded area, before/after/diff/%diff ----
    flood_before = wet_bl_count * cell_area_m2 / 1e6
    flood_after = wet_sc_count * cell_area_m2 / 1e6
    table_a_row = dict(
        scenario_tag=scenario_tag, baseline=baseline_tag, compare=compare_tag,
        flood_area_before_km2=flood_before, flood_area_after_km2=flood_after,
        diff_km2=flood_after - flood_before,
        pct_diff=(100 * (flood_after - flood_before) / flood_before) if flood_before else float("nan"),
    )
    print(f"\n[Table A -- total flooded area (depth > {WET_THRESHOLD_M} m), {baseline_tag} -> {compare_tag}]")
    print(f"  {'flood area before':<22}{'flood area after':>20}{'diff (km2)':>14}{'% diff':>10}")
    print(f"  {table_a_row['flood_area_before_km2']:<22.4f}{table_a_row['flood_area_after_km2']:>20.4f}"
          f"{table_a_row['diff_km2']:>+14.4f}{table_a_row['pct_diff']:>+9.1f}%")

    # ---- Table B: area per depth-magnitude bin, before/after/diff/%diff ----
    table_b_rows = []
    for k, lab in enumerate(DEPTH_MAG_BIN_LABELS):
        before = baseline_bin_counts[k] * cell_area_m2 / 1e6
        after = compare_bin_counts[k] * cell_area_m2 / 1e6
        diff = after - before
        table_b_rows.append(dict(
            scenario_tag=scenario_tag, baseline=baseline_tag, compare=compare_tag,
            depth_bin_m=lab, area_before_km2=before, area_after_km2=after, diff_km2=diff,
            pct_diff=(100 * diff / before) if before else (float("inf") if after else 0.0),
        ))
    table_b_df = pd.DataFrame(table_b_rows)

    print(f"\n[Table B -- area by depth bin (m), {baseline_tag} -> {compare_tag}]")
    print(f"  {'depth bin (m)':<16}{'before (km2)':>14}{'after (km2)':>14}{'diff (km2)':>14}{'% diff':>10}")
    for row in table_b_rows:
        pct = f"{row['pct_diff']:+.1f}%" if np.isfinite(row["pct_diff"]) else "n/a"
        print(f"  {row['depth_bin_m']:<16}{row['area_before_km2']:>14.4f}{row['area_after_km2']:>14.4f}"
              f"{row['diff_km2']:>+14.4f}{pct:>10}")

    # ---- CSVs ----
    table_a_csv = OUTPUT_DIR / f"flood_area_before_after_{scenario_tag}.csv"
    pd.DataFrame([table_a_row]).to_csv(table_a_csv, index=False)
    print(f"\nSaved: {table_a_csv}")
    table_b_csv = OUTPUT_DIR / f"flood_area_by_depth_bin_{scenario_tag}.csv"
    table_b_df.to_csv(table_b_csv, index=False)
    print(f"Saved: {table_b_csv}")

    summary_csv = OUTPUT_DIR / f"flood_comparison_stats_{scenario_tag}.csv"
    summary_row = dict(
        scenario_tag=scenario_tag, sim_baseline=baseline_tag, sim_compare=compare_tag,
        wet_baseline_cells=wet_bl_count, wet_compare_cells=wet_sc_count, wet_both_cells=wet_both_count,
        avoided_cells=avoided_count, area_avoided_km2=area_avoided_km2, pct_avoided_of_baseline=pct_avoided_of_baseline,
        new_cells=new_count, area_new_km2=area_new_km2,
        avg_depth_avoided_m=avg_depth_reduced, avg_depth_avoided_n_cells=count_depth_reduced,
        max_depth_avoided_m=max_depth_avoided_m,
        max_depth_avoided_x=min_xy[0] if min_xy else None, max_depth_avoided_y=min_xy[1] if min_xy else None,
        max_new_depth_m=max_new_depth_m,
        max_new_depth_x=new_max_xy[0] if new_max_xy else None, max_new_depth_y=new_max_xy[1] if new_max_xy else None,
    )
    with open(summary_csv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(summary_row.keys()))
        writer.writeheader()
        writer.writerow(summary_row)
    print(f"Saved: {summary_csv}")

    # ---- previews ----
    class_small = np.zeros(sc_small.shape, dtype="uint8")
    class_small[wet_sc_small & wet_bl_small] = 1
    class_small[wet_bl_small & ~wet_sc_small] = 2
    class_small[wet_sc_small & ~wet_bl_small] = 3

    def full_rc_to_preview_xy(rc):
        """Full-resolution (row, col) -> the matching (col, row) position in the DOWNSAMPLED
        preview array, for plotting a marker with ax.plot on the small preview image."""
        if rc is None:
            return None
        r, c = rc
        return c / width * small_w, r / height * small_h

    # --- Map 1 preview: depth difference, with the two marker dots ---
    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    im = ax.imshow(diff_small_masked, cmap=DIFF_CMAP, vmin=-vmax, vmax=vmax, origin=plot_origin)
    ax.set_title(f"Flood Depth Change ({compare_tag} minus {baseline_tag}) -- {scenario_tag}", fontsize=FS_TITLE)
    ax.axis("off")
    cbar = fig.colorbar(im, ax=ax, shrink=0.7,
                        label=f"Depth difference (m)\n<- {compare_tag} reduced depth    {compare_tag} increased depth  ->")
    cbar.set_label(cbar.ax.get_ylabel(), fontsize=FS_AXIS_LABEL)
    cbar.ax.tick_params(labelsize=FS_TICKS)
    add_stats_table(ax, STATS_TEXT_DEPTH)

    dot_handles = []
    avoided_xy_prev = full_rc_to_preview_xy(global_min_rc)
    if avoided_xy_prev is not None:
        ax.plot(*avoided_xy_prev, "o", color=DOT_MAX_AVOIDED_COLOR, markersize=DOT_MAX_AVOIDED_SIZE,
                markeredgecolor="black", markeredgewidth=0.6, zorder=10)
        dot_handles.append(plt.Line2D([0], [0], marker="o", color="w",
                           markerfacecolor=DOT_MAX_AVOIDED_COLOR, markeredgecolor="black",
                           markersize=9, label=f"Max depth avoided ({max_depth_avoided_m:.2f} m)"))
    new_depth_xy_prev = full_rc_to_preview_xy(global_new_max_rc)
    if new_depth_xy_prev is not None:
        ax.plot(*new_depth_xy_prev, "o", color=DOT_MAX_NEW_DEPTH_COLOR, markersize=DOT_MAX_NEW_DEPTH_SIZE,
                markeredgecolor="black", markeredgewidth=0.5, zorder=10)
        dot_handles.append(plt.Line2D([0], [0], marker="o", color="w",
                           markerfacecolor=DOT_MAX_NEW_DEPTH_COLOR, markeredgecolor="black",
                           markersize=6, label=f"Max new depth ({max_new_depth_m:.2f} m)"))
    if dot_handles:
        ax.legend(handles=dot_handles, loc="upper right", fontsize=FS_LEGEND, frameon=True)

    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"depth_diff_{scenario_tag}_preview.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- Map 2 preview: extent categories ---
    rgba_extent_small = np.zeros((*class_small.shape, 4), dtype="float32")
    for val, rgba in MAP2_COLORS.items():
        rgba_extent_small[class_small == val] = rgba

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    ax.imshow(rgba_extent_small, origin=plot_origin)
    ax.set_title(f"Flood extent difference -- {scenario_tag}", fontsize=FS_TITLE)
    ax.axis("off")
    legend_handles = [
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=MAP2_COLORS[1][:3], markersize=14, label="Wet in both"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=MAP2_COLORS[2][:3], markersize=14, label=f"New dry with {compare_tag}"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=MAP2_COLORS[3][:3], markersize=14, label=f"New wet with {compare_tag}"),
    ]
    ax.legend(handles=legend_handles, loc="upper left", bbox_to_anchor=(1.02, 1), frameon=True, fontsize=FS_LEGEND)
    add_stats_table(ax, STATS_TEXT_EXTENT)
    # NOTE: deliberately no tight_layout on this map. tight_layout re-balances the whole figure to
    # fit whatever the text now needs, so raising FS_TITLE/FS_LEGEND used to SHRINK THE MAP by the
    # same amount the text grew -- which is exactly why the title looked like it never changed.
    # Reserving the margins explicitly keeps the map the same size, so a bigger font is simply a
    # bigger font; bbox_inches="tight" below then widens the saved image to fit the legend.
    fig.subplots_adjust(left=0.02, right=MAP_LEGEND_RIGHT, top=0.94, bottom=0.02)
    plt.savefig(OUTPUT_DIR / f"extent_diff_{scenario_tag}_preview.png", dpi=200, bbox_inches="tight")
    plt.show()

    # --- Map 3 preview: combined -- grey/red plain, green-red gradient for wet-in-both ---
    rgba_combined_small = cmap(norm(diff_small)).astype("float32")
    rgba_combined_small[..., 3] = 0.85
    rgba_combined_small[class_small == 0] = MAP3_PLAIN_COLORS[0]
    rgba_combined_small[class_small == 2] = MAP3_PLAIN_COLORS[2]
    rgba_combined_small[class_small == 3] = MAP3_PLAIN_COLORS[3]

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    ax.imshow(rgba_combined_small, origin=plot_origin)
    ax.set_title(f"Combined flood depth + extent difference -- {scenario_tag}", fontsize=FS_TITLE)
    ax.axis("off")
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    sm.set_array([])
    cbar3 = fig.colorbar(sm, ax=ax, shrink=0.6,
                         label=f"Depth difference, wet in both (m)\n<- {compare_tag} reduced depth    {compare_tag} increased depth  ->")
    cbar3.set_label(cbar3.ax.get_ylabel(), fontsize=FS_AXIS_LABEL)
    cbar3.ax.tick_params(labelsize=FS_TICKS)
    legend_handles3 = [
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=MAP3_PLAIN_COLORS[2][:3], markersize=14, label=f"New dry with {compare_tag} -- grey"),
        plt.Line2D([0], [0], marker="s", color="w", markerfacecolor=MAP3_PLAIN_COLORS[3][:3], markersize=14, label=f"New wet with {compare_tag} -- red"),
    ]
    ax.legend(handles=legend_handles3, loc="upper center", bbox_to_anchor=(0.5, -0.02), frameon=True, fontsize=FS_LEGEND, ncol=2)
    add_stats_table(ax, STATS_TEXT_COMBINED)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"combined_diff_{scenario_tag}_preview.png", dpi=200, bbox_inches="tight")
    plt.show()

    return table_a_row, table_b_df


## Run it -- process every pair in `SIM_PAIRS`

Calls `process_scenario_pair` once per pair, then combines every pair's Table A and
Table B into two "all pairs processed this run" CSVs, so you have one place to see every
scenario compared in this run side by side, on top of the per-pair CSVs saved inside the
function itself.


In [ ]:
all_table_a_rows = []
all_table_b_frames = []

for _baseline_tag, _compare_tag in SIM_PAIRS:
    _table_a_row, _table_b_df = process_scenario_pair(_baseline_tag, _compare_tag)
    all_table_a_rows.append(_table_a_row)
    all_table_b_frames.append(_table_b_df)

table_a_all_df = pd.DataFrame(all_table_a_rows)
table_b_all_df = pd.concat(all_table_b_frames, ignore_index=True)

_tag_join = "_".join(f"{b}-{c}" for b, c in SIM_PAIRS)
table_a_all_csv = OUTPUT_DIR / f"flood_area_before_after_ALL_{_tag_join}.csv"
table_a_all_df.to_csv(table_a_all_csv, index=False)
table_b_all_csv = OUTPUT_DIR / f"flood_area_by_depth_bin_ALL_{_tag_join}.csv"
table_b_all_df.to_csv(table_b_all_csv, index=False)

print(f"\n{'=' * 78}\nALL {len(SIM_PAIRS)} PAIRS PROCESSED THIS RUN\n{'=' * 78}")
print("\nTable A (total flooded area):")
with pd.option_context("display.width", 200):
    print(table_a_all_df.to_string(index=False))
print(f"\nSaved: {table_a_all_csv}")

print("\nTable B (area by depth bin) -- pivoted, diff_km2 per bin per scenario:")
_pivot_b = table_b_all_df.pivot(index="scenario_tag", columns="depth_bin_m", values="diff_km2")
_pivot_b = _pivot_b.reindex(index=[f"{b}_vs_{c}" for b, c in SIM_PAIRS],
                            columns=DEPTH_MAG_BIN_LABELS)
with pd.option_context("display.width", 200):
    print(_pivot_b.to_string())
print(f"\nSaved: {table_b_all_csv}")


## Sanity checks before trusting these maps

- **Depth range printed in Step 2** should roughly match what you've seen for this
  scenario in QGIS before (e.g. up to a few metres near the channel).
- **Wet-cell counts printed in Step 3** should be in the same ballpark for both
  scenarios -- if one is wildly larger than the other, double check both TIFFs are
  really the same event (rainfall / boundary conditions) and domain.
- **The statistics** give you the average depth reduction, the avoided/new
  areas (with a percentage), and the max/min single-cell differences directly -- use these
  numbers rather than eyeballing the map. Their locations are now marked on Map 1 (the olive
  and orange dots) rather than printed as coordinates in its on-map table.
- **Table A and Table B** are the two summary tables asked for: total flooded area before/after/
  diff/%diff, and the same split by depth-magnitude bin -- both printed per pair and combined
  across every scenario processed in this run.
- If avoided/new cells look suspiciously speckled everywhere rather than concentrated
  near the floodplain edges, that's typical small-isolated-patch noise -- worth revisiting a minimum-mapping-unit filter for these
  specific outputs if it shows up here too.


## Binned depth-difference map

The gradient map answers *how much* the depth changed, but two things are hard to read off it: where the change crosses a threshold you care about, and how deep the **newly flooded** cells actually are -- they were all one flat red regardless of whether they hold 5 cm or 2 m.

This bins the signed difference into depth classes. Because the reads are zero-filled (dry = 0 m), one signed scale covers all three cases at once:

| case | value | reads as |
|---|---|---|
| wet in both | `compare - baseline` | the real change in depth |
| avoided (wet -> dry) | negative, = how deep it **used to be** | green, graded by what was drained |
| newly wet (dry -> wet) | positive, = how deep it **now is** | red, graded by what appeared |

So a cell newly flooded to 2 m is dark red while one newly flooded to 5 cm is the palest red, and draining a 2 m pool reads darker green than drying a 5 cm puddle. Green stays good, red stays bad. Writes a classified GeoTIFF with a matching `.qml` (areas in the legend labels), a CSV, and a preview -- **for every pair in `SIM_PAIRS`**, same as the maps above.

Classes run from `reduced >= 2 m` down through `no change (< 0.1 m)` up to `increased >= 2 m` -- 11 in total. The outermost class on each side is open-ended, so a 7 m change is never mislabelled as sitting in a bounded bin. Change `DIFF_BIN_EDGES` to shift the thresholds; labels, colours, CSV and QML all follow. This is a genuinely different classification from Table B above (which bins each scenario's own absolute depth, not the signed change between them) -- kept as its own map since it answers a different question.


In [ ]:
# ============================================================
# Binned depth-difference map (+ classified GeoTIFF and QGIS style)
#
# One SIGNED scale covers all three cases, because reads are zero-filled (dry = 0 m depth):
#   wet in both      -> diff = compare - baseline, the real change in depth
#   avoided (wet->dry) -> negative, magnitude = how deep the water USED to be
#   newly wet (dry->wet) -> positive, magnitude = how deep the new water IS
#
# That fixes the flat-red problem: a cell newly flooded to 2 m now lands in a dark red bin
# while one newly flooded to 5 cm lands in the palest red, instead of both being one colour.
# Same on the green side -- draining a 2 m pool reads darker than drying a 5 cm puddle.
# ============================================================
from xml.sax.saxutils import escape
from matplotlib.patches import Patch

# Magnitude breaks, mirrored either side of zero. Same class edges as the flood-depth maps.
# The OUTERMOST class is open-ended and labelled ">= <second-to-last edge> m", so a change of
# 7 m is never mislabelled as belonging to a bounded bin. Add 5.0 -> [..., 2.0, 5.0] if you
# later want a separate "> 5 m" class back.
DIFF_BIN_EDGES = [0.1, 0.2, 0.5, 1.0, 2.0, 5.0]
DIFF_NOCHANGE_M = DIFF_BIN_EDGES[0]      # |diff| below this counts as no meaningful change

# green = reduction (good), red = increase (bad), pale grey = no change
_GREEN_RAMP = ["#00441b", "#006d2c", "#238b45", "#41ab5d", "#74c476", "#bae4b3"]  # deepest -> palest
_RED_RAMP   = ["#fcbba1", "#fc9272", "#fb6a4a", "#ef3b2c", "#cb181d", "#67000d"]  # palest -> deepest
NOCHANGE_COLOR = "#f0f0f0"

# Build the classes from DIFF_BIN_EDGES. With edges [0.1, 0.2, 0.5, 1, 2, 5] the bounded bins
# are 0.1-0.2 ... 2-5, and the outermost class catches everything beyond, labelled ">= 2 m".
_bounded = list(zip(DIFF_BIN_EDGES[:-1], DIFF_BIN_EDGES[1:]))   # [(0.1,0.2), ..., (2,5)]
_outer_lbl = f"{DIFF_BIN_EDGES[-2]:g} m"                        # e.g. "2 m" -> ">= 2 m"
n_side = len(_bounded)

GREENS = _GREEN_RAMP[len(_GREEN_RAMP) - n_side:]
REDS = _RED_RAMP[:n_side]

_labels = ([f"reduced >= {_outer_lbl}"]
          + [f"reduced {lo:g} - {hi:g} m" for lo, hi in reversed(_bounded[:-1])]
          + [f"no change (< {DIFF_NOCHANGE_M:g} m)"]
          + [f"increased {lo:g} - {hi:g} m" for lo, hi in _bounded[:-1]]
          + [f"increased >= {_outer_lbl}"])
_colors = GREENS + [NOCHANGE_COLOR] + REDS

# class k covers (_upper[k-1], _upper[k]]; first and last are open-ended
_upper = ([-DIFF_BIN_EDGES[-2]]
         + [-e for e in reversed(DIFF_BIN_EDGES[:-2])]
         + [DIFF_BIN_EDGES[0]]
         + DIFF_BIN_EDGES[1:-1]
         + [np.inf])
CLASS_CODES = list(range(1, len(_labels) + 1))
assert len(_labels) == len(_colors) == len(_upper), (len(_labels), len(_colors), len(_upper))


def classify_diff(d, valid):
    """Signed difference -> class code 1..N; 0 where neither scenario was wet."""
    out = np.zeros(d.shape, dtype="uint8")
    assigned = np.zeros(d.shape, dtype=bool)
    for code_, up in zip(CLASS_CODES, _upper):
        in_cls = valid & ~assigned & ((d <= up) if np.isfinite(up) else valid & ~assigned)
        out[in_cls] = code_
        assigned |= in_cls
    return out


def process_binned_diff_map(baseline_tag, compare_tag):
    """Signed depth-change classification for one pair -- reopens the rasters itself (they're
    closed at the end of process_scenario_pair), so it runs standalone in any position."""
    sc_tif = _require_map(compare_tag)
    bl_tif = _require_map(baseline_tag)
    scenario_tag = f"{baseline_tag}_vs_{compare_tag}"

    _sc = rasterio.open(sc_tif)
    _bl_raw = rasterio.open(bl_tif)
    _same_grid = (_sc.transform == _bl_raw.transform and _sc.crs == _bl_raw.crs
                 and _sc.width == _bl_raw.width and _sc.height == _bl_raw.height)
    _bl = _bl_raw if _same_grid else WarpedVRT(
        _bl_raw, crs=_sc.crs, transform=_sc.transform,
        width=_sc.width, height=_sc.height, resampling=Resampling.bilinear)
    transform, crs = _sc.transform, _sc.crs
    width, height = _sc.width, _sc.height
    plot_origin = "upper" if transform.e < 0 else "lower"
    cell_area_m2 = abs(transform.a * transform.e)

    binned_out = OUTPUT_DIR / f"depth_diff_binned_{scenario_tag}.tif"
    binned_profile = dict(driver="GTiff", height=height, width=width, count=1, dtype="uint8",
                          crs=crs, transform=transform, nodata=0, compress="deflate",
                          tiled=True, blockxsize=256, blockysize=256, BIGTIFF="YES")

    counts = {c: 0 for c in CLASS_CODES}
    try:
        with rasterio.open(binned_out, "w", **binned_profile) as dst:
            for _, window in _sc.block_windows(1):
                sc_b = read_block_zero_filled(_sc, window)
                bl_b = read_block_zero_filled(_bl, window)
                valid = (sc_b > WET_THRESHOLD_M) | (bl_b > WET_THRESHOLD_M)
                cls = classify_diff(sc_b - bl_b, valid)
                dst.write(cls, 1, window=window)
                for c in CLASS_CODES:
                    counts[c] += int((cls == c).sum())
    finally:
        if not _same_grid:
            _bl.close()
        _sc.close()
        _bl_raw.close()

    areas = {c: counts[c] * cell_area_m2 / 1e6 for c in CLASS_CODES}
    total = sum(areas.values())

    print(f"--- binned depth difference: {compare_tag} vs {baseline_tag} ---")
    print(f"Saved: {binned_out}\n")
    print(f"  {'class':<26} {'cells':>14} {'km2':>10} {'% of changed area':>18}")
    for c, lab in zip(CLASS_CODES, _labels):
        print(f"  {lab:<26} {counts[c]:>14,} {areas[c]:>10,.2f} {100 * areas[c] / max(total, 1e-12):>17.1f}%")
    print(f"  {'TOTAL (wet in either)':<26} {sum(counts.values()):>14,} {total:>10,.2f}")

    reduced = sum(areas[c] for c in CLASS_CODES[:len(GREENS)])
    increased = sum(areas[c] for c in CLASS_CODES[-len(REDS):])
    print(f"\n  net: {reduced:,.2f} km2 shallower vs {increased:,.2f} km2 deeper "
          f"({reduced - increased:+,.2f} km2 in favour of {compare_tag if reduced > increased else baseline_tag})")

    binned_csv = OUTPUT_DIR / f"depth_diff_binned_{scenario_tag}.csv"
    with open(binned_csv, "w", newline="") as f:
        w_ = csv.writer(f)
        w_.writerow(["class_code", "label", "n_cells", "area_km2", "pct_of_changed_area"])
        for c, lab in zip(CLASS_CODES, _labels):
            w_.writerow([c, lab, counts[c], round(areas[c], 4), round(100 * areas[c] / max(total, 1e-12), 3)])
    print(f"Saved: {binned_csv}")

    # --- QGIS style, same basename as the tif so it is applied automatically on load ---
    entries = "\n          ".join(
        f'<paletteEntry value="{c}" color="{col}" alpha="255" label="{escape(f"{lab}  |  {areas[c]:,.2f} km2")}"/>'
        for c, col, lab in zip(CLASS_CODES, _colors, _labels)
    )
    qml = f"""<!DOCTYPE qgis PUBLIC 'http://mrcc.com/qgis.dtd' 'SYSTEM'>
<qgis version="3.28" styleCategories="AllStyleCategories">
  <pipe>
    <rasterrenderer type="paletted" band="1" opacity="1" alphaBand="-1" nodataColor="">
      <rasterTransparency/>
      <colorPalette>
          {entries}
      </colorPalette>
    </rasterrenderer>
    <brightnesscontrast brightness="0" contrast="0"/>
    <huesaturation colorizeOn="0" saturation="0" grayscaleMode="0"/>
    <rasterresampler/>
  </pipe>
  <blendMode>0</blendMode>
</qgis>
"""
    qml_path = binned_out.with_suffix(".qml")
    with open(qml_path, "w", encoding="utf-8") as f:
        f.write(qml)
    print(f"Saved QGIS style: {qml_path}")

    # --- preview from a fresh downsampled read (this cell is self-contained) ---
    small_h = max(1, height // PREVIEW_DOWNSAMPLE)
    small_w = max(1, width // PREVIEW_DOWNSAMPLE)
    with rasterio.open(sc_tif) as _s:
        sc_small = _s.read(1, out_shape=(small_h, small_w), resampling=Resampling.average,
                            masked=True).filled(0.0)
    with rasterio.open(bl_tif) as _s:
        bl_small = _s.read(1, out_shape=(small_h, small_w), resampling=Resampling.average,
                              masked=True).filled(0.0)
    sc_small = np.where(sc_small < 0, 0.0, sc_small).astype("float32")
    bl_small = np.where(bl_small < 0, 0.0, bl_small).astype("float32")

    valid_small = (sc_small > WET_THRESHOLD_M) | (bl_small > WET_THRESHOLD_M)
    cls_small = classify_diff(sc_small - bl_small, valid_small)

    rgba = np.zeros((*cls_small.shape, 4), dtype="float32")
    for c, col in zip(CLASS_CODES, _colors):
        rgba[cls_small == c] = mcolors.to_rgba(col)

    fig, ax = plt.subplots(figsize=MAP_FIGSIZE)
    ax.imshow(rgba, origin=plot_origin)
    ax.set_title(f"Flood depth change by class ({compare_tag} minus {baseline_tag}) -- {scenario_tag}",
                fontsize=FS_TITLE)
    ax.axis("off")
    handles = [Patch(facecolor=col, edgecolor="0.6", linewidth=0.4,
                     label=f"{lab}  --  {areas[c]:,.2f} km2")
              for c, col, lab in zip(CLASS_CODES, _colors, _labels)]
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=FS_LEGEND,
              title="Depth change", title_fontsize=FS_LEGEND_TITLE, frameon=True)
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / f"depth_diff_binned_{scenario_tag}_preview.png", dpi=200, bbox_inches="tight")
    plt.show()


for _baseline_tag, _compare_tag in SIM_PAIRS:
    process_binned_diff_map(_baseline_tag, _compare_tag)
